# Chapter 16: Classification Ratemaking with GLMs

Code of Chapter 16 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## Exercise 2: Is red a risk?

In [ ]:
from gitriangle import synthetic_history
from giglm import policy_data, glm_fit, design, lr_test
policies, claims, _ = synthetic_history()
pol, _ = policy_data(policies, claims, [2023, 2024, 2025])
B = {"region": "South", "vehicle": "A", "color": "black"}
f1 = glm_fit(design(pol, ["region", "vehicle"], B), pol.n_claims, "poisson")
f2 = glm_fit(design(pol, ["region", "vehicle", "color"], B), pol.n_claims, "poisson")
lr_test(f1, f2)                                   # (3.57, 3, 0.31)

## 16.7 Python Walkthrough

In [ ]:
import numpy as np
from gitriangle import synthetic_history
from giglm import (policy_data, design, glm_fit, relativities, lr_test,
                   lift_table, gini, double_lift)

policies, claims, _ = synthetic_history()
pol, cl = policy_data(policies, claims, [2023, 2024, 2025])   # trended to 2028, color added
B = {"region": "South", "vehicle": "A", "color": "black"}

# Section 16.2: frequency, severity and Tweedie models
freq = glm_fit(design(pol, ["region", "vehicle"], B), pol.n_claims, "poisson",
               offset=np.log(pol.exposure))
relativities(freq).round(3)                       # 0.092, 1.168, 0.878, 1.108, 1.436
sev = glm_fit(design(cl, ["region", "vehicle"], B), cl.amount, "gamma")
tw = glm_fit(design(pol, ["region", "vehicle"], B), pol.losses, "tweedie", p=1.5)

# Section 16.4: holdout validation on policy year 2025
tr, te = pol[pol.policy_year < 2025], pol[pol.policy_year == 2025]
fa = glm_fit(design(tr, ["region", "vehicle"], B), tr.n_claims, "poisson")
pa = np.exp(design(te, ["region", "vehicle"], B)[fa["coef"].index] @ fa["coef"])
lift_table(pa, te.n_claims, te.exposure)
gini(pa, te.n_claims, te.exposure)                # 0.071